# Gated Recurrent Unit (GRU) - PyTorch

**Goal:** Classify IMDB movie-review sentiment.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Reset and update gates provide compact recurrent memory.
- **Where it is used:** sequence modeling with fewer parameters than LSTM.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Gated Recurrent Unit: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['input', 'gates', 'state']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = 1/(1+np.exp(-1.5*x))
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.42,.58])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['reset', 'update'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Synthetic sequence dataset: class 1 has stronger late positive signal.
samples, seq_len, features = 1400, 30, 3
X = np.random.normal(size=(samples, seq_len, features)).astype("float32")
signal = X[:, -10:, 0].mean(axis=1) + 0.5 * X[:, :10, 1].mean(axis=1)
y = (signal > 0.05).astype("int64")
train_loader = DataLoader(
    TensorDataset(torch.tensor(X[:1000]), torch.tensor(y[:1000])),
    batch_size=64,
    shuffle=True,
)
test_x = torch.tensor(X[1000:], device=device)
test_y = torch.tensor(y[1000:], device=device)


In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, input_dim: int = 3, hidden_dim: int = 32):
        super().__init__()
        self.recurrent = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=False,
        )
        self.classifier = nn.Linear(hidden_dim * 1, 2)

    def forward(self, x):
        output, _ = self.recurrent(x)
        return self.classifier(output[:, -1, :])


model = GRUClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


In [ ]:
for epoch in range(10):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        accuracy = (model(test_x).argmax(dim=1) == test_y).float().mean().item()
    print(f"epoch={epoch+1:02d} accuracy={accuracy:.3f}")
